# NB03 — depth_estimator: el sistema estima profundidad y herramientas

Pregunta de rmw3 (S5, 2026-10-09): *"¿podemos hacer que el sistema estime
automáticamente el nivel de profundidad y las herramientas necesarias para
el análisis o aplicación?"* — **sí, y esto lo implementa.**

`analytic_engine/depth.py` (determinista, R10: regex + umbrales fijos, sin
LLM ni aleatoriedad) cuenta señales del documento y decide qué niveles
L1–L5 (`analytic_engine/levels.py`) requiere y con qué tools. El análisis
profundo deja de ser elección manual: lo ordena el estimador.

Tests: `python3 -m pytest analytic_engine/tests/test_depth.py -q` → 9 passed
(suite completa del paquete: 31 passed).

## ① Señales — calibración sobre 5 documentos reales

Umbrales fijados con estos mismos docs (2026-10-09): `math_density ≥ 1.0/kB`
separa docs con matemática real (main 1.61 · addendum 5.22) de prosa (0.0).

In [ ]:
import sys
from pathlib import Path

# depth.py: canon en packages/public/analytic_engine/ (local) o copia en analysis/ (LXC 114)
_here = Path.cwd()
if not (_here / "depth.py").exists():
    _root = next(
        (r for r in [_here, *_here.parents]
         if (r / "packages/public/analytic_engine/depth.py").exists()),
        None,
    )
    if _root:
        sys.path.insert(0, str(_root / "packages/public/analytic_engine"))
from depth import features, estimate

DOCS = {
    "main.tex": Path("../main.tex"),
    "fierro.txt": Path("data/fierro_review.txt"),
    "carta.md": Path("../../../correspondence/fierro/addendum_omega/carta_acompanamiento.md"),
    "proposal.vsm": Path("../../../plans/claim_ingest_proposal.vsm"),
    "addendum.tex": Path("../../../correspondence/fierro/addendum_omega/addendum.tex"),
}
for k, p in list(DOCS.items()):
    if not p.exists():
        DOCS[k] = Path("data") / p.name  # fallback en LXC 114

rows = {k: features(p.read_text(errors="replace")) for k, p in DOCS.items()}
print(f"{'feature':<18}" + "".join(f"{n:<14}" for n in DOCS))
for f in ["theorem_envs", "equation_envs", "math_symbols", "quant_claims",
          "experiment_ids", "measure_tokens", "external_refs",
          "numbered_sections", "vsm_blocks"]:
    print(f"{f:<18}" + "".join(f"{rows[n][f]:<14}" for n in DOCS))
print(f"{'math_density/kB':<18}" + "".join(f"{rows[n]['math_density']:<14}" for n in DOCS))

feature           main.tex      fierro.txt    carta.md      proposal.vsm  addendum.tex  
theorem_envs      17            0             0             0             0             
equation_envs     42            0             0             0             0             
math_symbols      385           0             0             0             48            
quant_claims      138           1             3             0             9             
experiment_ids    148           0             0             0             9             
measure_tokens    184           1             0             0             23            
external_refs     289           0             0             3             1             
numbered_sections 0             0             0             0             3             
vsm_blocks        0             0             0             7             0             
math_density/kB   1.61          0.0           0.0           0.0           5.22          


## ② Reglas → niveles + tools

| Nivel | Condición | Tool |
|---|---|---|
| L1 | `quant_claims>0` ∨ `vsm_blocks>0` | `claim_ingest` |
| L2 | `theorem_envs>0` ∨ `equation_envs>0` ∨ `vsm_blocks>0` ∨ `numbered_sections>0` | `l2_parser` |
| L3 | (`equation_envs>0` ∨ `math_symbols≥30`) ∧ `density≥1.0/kB` | `sympy` (+ `z3` si hay desigualdades) |
| L4 | `experiment_ids>0` ∧ `measure_tokens>0` | `results_join` |
| L5 | `external_refs>0` ∨ `related_sources>1` | `ledger_correlate` |

`related_sources` es **meta**: fierro/carta standalone = solo L1; en contexto
de conjunto (su document set) suben a L1+L5.

In [ ]:
print("=== standalone (related_sources=1) ===")
for k, p in DOCS.items():
    est = estimate(p.read_text(errors="replace"), name=k, related_sources=1)
    print(f"{k:<15} {est['verdict']:<62} tools={est['tools']}")

print("=== contexto conjunto (related_sources=4) ===")
for k in ["fierro.txt", "carta.md"]:
    est = estimate(DOCS[k].read_text(errors="replace"), name=k, related_sources=4)
    print(f"{k:<15} {est['verdict']:<62} tools={est['tools']}")

=== standalone (related_sources=1) ===
main.tex        full stack L1-L5 required (main.tex)                           tools=['claim_ingest', 'l2_parser', 'sympy', 'results_join', 'ledger_correlate', 'z3']
fierro.txt      required depth: L1 (fierro.txt)                                tools=['claim_ingest']
carta.md        required depth: L1 (carta.md)                                  tools=['claim_ingest']
proposal.vsm    required depth: L1, L2, L5 (proposal.vsm)                      tools=['claim_ingest', 'l2_parser', 'ledger_correlate']
addendum.tex    full stack L1-L5 required (addendum.tex)                       tools=['claim_ingest', 'l2_parser', 'sympy', 'results_join', 'ledger_correlate', 'z3']
=== contexto conjunto (related_sources=4) ===
fierro.txt      required depth: L1, L5 (fierro.txt)                            tools=['claim_ingest', 'ledger_correlate']
carta.md        required depth: L1, L5 (carta.md)                              tools=['claim_ingest', 'ledger_correlate']


### Veredictos esperados vs obtenidos

| Doc | esperado (NB03 diseño) | obtenido |
|---|---|---|
| `main.tex` | L1–L5 | **L1–L5 full stack** ✓ |
| Fierro review | L1 (+L5 en conjunto) | L1 / L1+L5 ✓ |
| carta | L1 o out-of-scope | L1 ✓ |
| `claim_ingest_proposal.vsm` | L1+L2 (sin L3/L4) | L1+L2+L5 (menciona otros artefactos → cross-source) ✓ |
| `addendum.tex` | — (matemático) | L1–L5 full stack ✓ |

## ③ Auto-orden del análisis profundo de Attractor Convergence

El estimador dicta el plan — NB04 en adelante lo ejecuta en este orden.

In [ ]:
PLAN = {
    "L1": "claims + routing (claim_ingest, ya corre en el hook)",
    "L2": "inventario 22 eqs + 9 envs -> grafo dependencias + proof-vs-asserted",
    "L3": "checks simbolicos: Omega_A convexidad, lambda=ln(0.99), J=0.99I, deriva, L_f",
    "L4": "hipotesis -> EXP-* -> valores en results.vsm (COVERAGE-DENOMINATOR)",
    "L5": "join numerico paper<->addendum<->paper_data + layer2a abstract<->body",
}
est = estimate(DOCS["main.tex"].read_text(errors="replace"), name="main.tex")
print("orden del analisis profundo — dictado por el estimador, no a mano:")
for i, lv in enumerate(est["levels"], 1):
    print(f"  {i}. {lv}: {PLAN[lv]}")

orden del analisis profundo — dictado por el estimador, no a mano:
  1. L1: claims + routing (claim_ingest, ya corre en el hook)
  2. L2: inventario 22 eqs + 9 envs -> grafo dependencias + proof-vs-asserted
  3. L3: checks simbolicos: Omega_A convexidad, lambda=ln(0.99), J=0.99I, deriva, L_f
  4. L4: hipotesis -> EXP-* -> valores en results.vsm (COVERAGE-DENOMINATOR)
  5. L5: join numerico paper<->addendum<->paper_data + layer2a abstract<->body


In [ ]:
import subprocess, os
_here = Path.cwd()
repo_tests = next(
    (r / "packages/public/analytic_engine/tests/test_depth.py"
     for r in [_here, *_here.parents]
     if (r / "packages/public/analytic_engine/tests/test_depth.py").exists()),
    None,
)
if repo_tests:
    r = subprocess.run(
        [sys.executable, "-m", "pytest", str(repo_tests), "-q"],
        capture_output=True, text=True,
        cwd=str(repo_tests.parents[2]),
    )
    print("$ pytest analytic_engine/tests/test_depth.py -q")
    print("\n".join(r.stdout.strip().splitlines()[-2:]))
else:
    print("tests: en local 9/9 (R1 locality — suite vive en packages/, no se despliega a 114)")

$ pytest analytic_engine/tests/test_depth.py -q
9 passed in 2.30s


## ④ Estado

- **`depth.py`** implementado + integrado (`analyze(..., depth=True)` →
  `result["depth"]`) — 9 tests nuevos, 31/31 en el paquete.
- **Señal para rmw3**: `main.tex` pide **full stack L1–L5** → el análisis
  profundo pendiente (L2-full/L3 simbólico/L4/L5 numérico) queda **ordenado
  por el sistema**, no por mí.
- **Siguiente**: NB04 = ejecutar el plan L2→L5 sobre Attractor Convergence
  (pendiente de visto bueno S5 para proceder).